In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/validation/ibm'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Age–Size DeepRV Validation Analysis

This notebook compares the age–size Transformer DeepRV IBM with the exact prior-GP reference. It loads the raw chunk files directly, checks that all 5,000 parameter combinations and 50 replicates are present, and then aggregates over replicates before comparing the two independently simulated methods.

The analysis is restricted to the five statistics used by the completed ABC. It does not perform the offline ABC check or treat replicate identifiers as paired random draws.

In [ ]:
from pathlib import Path
import json
import time
import textwrap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import spearmanr

benchmark_name = "benchmark_age_size_5k_r50_transformer_1_512"
candidates = [GENERATED / "soay/age_size/validation/ibm"]
results_directory = next((path.resolve() for path in candidates if (path / "benchmark_design.npz").exists()), None)
if results_directory is None:
    raise FileNotFoundError(f"Could Not Find {benchmark_name}")
analysis_directory = results_directory / "analysis_outputs"
analysis_directory.mkdir(exist_ok=True)
minimum_replicates = 3

with np.load(results_directory / "benchmark_design.npz") as design:
    number_parameters = int(design["number_parameters"])
    number_replicates = int(design["number_replicates"])
    years = design["years"].astype(int)
    summary_names = design["summary_names"].astype(str)
    parameter_names = design["parameter_names"].astype(str)
    parameter_values = design["parameter_values"]

summary_labels = {
    "emd_breeder_next_size_distribution": "Breeder Next Size EMD",
    "chi2_total_survivors_by_size_age": "Survivors By Size And Age Chi-Squared",
    "chi2_total_breeders_by_size": "Breeders By Size Chi-Squared",
    "chi2_total_recruits_by_age": "Recruits By Age Chi-Squared",
    "bhattacharyya_nonbreeder_next_size_distribution": "Nonbreeder Next Size Bhattacharyya",
}
plt.style.use("seaborn-v0_8-white")

def style_axis(axis):
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

print(f"Results Directory: {results_directory}")
print(f"Design: {number_parameters:,} Parameters × {number_replicates} Replicates × {len(years)} Years")

## Load And Validate Raw Chunks

The loader places each result using its parameter and replicate identifiers. It rejects missing tasks, duplicates, nonempty error messages, or mismatched designs.

In [ ]:
def load_method(method):
    chunk_paths = sorted((results_directory / method).glob("chunk_*.npz"))
    if not chunk_paths:
        raise FileNotFoundError(f"No Chunk Files Found For {method}")
    expected = number_parameters * number_replicates
    with np.load(chunk_paths[0]) as first:
        arrays = {
            "distances": np.empty((expected,) + first["distances"].shape[1:], dtype=first["distances"].dtype),
            "population_sizes": np.empty((expected,) + first["population_sizes"].shape[1:], dtype=first["population_sizes"].dtype),
            "status": np.empty(expected, dtype=first["status"].dtype),
            "extinction_year": np.empty(expected, dtype=first["extinction_year"].dtype),
            "seconds": np.empty(expected, dtype=first["seconds"].dtype),
        }
    seen = np.zeros(expected, dtype=bool)
    for chunk_path in chunk_paths:
        with np.load(chunk_path) as chunk:
            parameter_id = chunk["parameter_id"].astype(int)
            replicate_id = chunk["replicate_id"].astype(int)
            flat_id = parameter_id * number_replicates + replicate_id
            if np.any(flat_id < 0) or np.any(flat_id >= expected):
                raise ValueError(f"Out-Of-Range Task In {chunk_path.name}")
            if len(np.unique(flat_id)) != len(flat_id) or np.any(seen[flat_id]):
                raise ValueError(f"Duplicate Task In {chunk_path.name}")
            if np.any(chunk["error"] != ""):
                raise ValueError(f"Simulation Error In {chunk_path.name}")
            for name in arrays:
                arrays[name][flat_id] = chunk[name]
            seen[flat_id] = True
    if not np.all(seen):
        raise ValueError(f"{method} Is Missing {np.sum(~seen):,} Tasks")
    for name, values in arrays.items():
        arrays[name] = values.reshape((number_parameters, number_replicates) + values.shape[1:])
    arrays["number_chunks"] = len(chunk_paths)
    return arrays

load_started = time.perf_counter()
exact = load_method("exact_gp")
deeprv = load_method("deeprv")
exact_info = json.loads((results_directory / "exact_gp" / "run_info.json").read_text())
deeprv_info = json.loads((results_directory / "deeprv" / "run_info.json").read_text())
if exact_info["design_sha256"] != deeprv_info["design_sha256"]:
    raise ValueError("Exact-GP And DeepRV Runs Used Different Designs")
print(f"Loaded And Validated {2 * number_parameters * number_replicates:,} Simulations In {time.perf_counter() - load_started:.1f} Seconds")
print(f"Exact-GP Chunks: {exact['number_chunks']:,}")
print(f"DeepRV Chunks: {deeprv['number_chunks']:,}")
print(f"Distance Shape Per Method: {exact['distances'].shape}")

## Parameter-Level Accuracy

For every parameter combination, year, and statistic, the notebook estimates the mean and standard deviation across finite replicates. Extinct trajectories contribute finite statistics until extinction and infinite values thereafter. DeepRV overflow trajectories are nonfinite throughout and are excluded. At least three finite replicates per method are required.

The complete initial population is age zero, so breeder and recruit summaries are structurally constant in year one. Their year-one correlation, scale-normalised difference, SD ratio, and Monte Carlo agreement are therefore undefined and appear as blank cells.

The Monte Carlo standard error is $\sqrt{s_{\mathrm{exact}}^2/n_{\mathrm{exact}} + s_{\mathrm{DeepRV}}^2/n_{\mathrm{DeepRV}}}$.

In [ ]:
def finite_mean_sd(values):
    finite = np.isfinite(values)
    count = finite.sum(axis=1)
    total = np.where(finite, values, 0).sum(axis=1)
    mean = np.divide(total, count, out=np.full(total.shape, np.nan, dtype=float), where=count > 0)
    centred = np.where(finite, values - mean[:, None], 0)
    variance = np.divide((centred**2).sum(axis=1), count - 1, out=np.full(total.shape, np.nan, dtype=float), where=count > 1)
    return mean, np.sqrt(variance), count

def safe_spearman(x, y):
    if len(x) < 3 or np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan
    return float(spearmanr(x, y).statistic)

exact_mean, exact_sd, exact_count = finite_mean_sd(exact["distances"])
deeprv_mean, deeprv_sd, deeprv_count = finite_mean_sd(deeprv["distances"])
difference_se = np.sqrt(exact_sd**2 / exact_count + deeprv_sd**2 / deeprv_count)

rows = []
for year_index, year in enumerate(years):
    for statistic_index, statistic in enumerate(summary_names):
        exact_values = exact_mean[:, year_index, statistic_index]
        deeprv_values = deeprv_mean[:, year_index, statistic_index]
        valid = np.isfinite(exact_values) & np.isfinite(deeprv_values) & (exact_count[:, year_index, statistic_index] >= minimum_replicates) & (deeprv_count[:, year_index, statistic_index] >= minimum_replicates)
        exact_valid = exact_values[valid]
        deeprv_valid = deeprv_values[valid]
        difference = deeprv_valid - exact_valid
        exact_sd_values = exact_sd[valid, year_index, statistic_index]
        standardised_difference = np.divide(np.abs(difference), exact_sd_values, out=np.full(difference.shape, np.nan), where=np.isfinite(exact_sd_values) & (exact_sd_values > 0))        
        sd_valid = valid & np.isfinite(exact_sd[:, year_index, statistic_index]) & np.isfinite(deeprv_sd[:, year_index, statistic_index]) & (exact_sd[:, year_index, statistic_index] > 0)
        sd_ratio = deeprv_sd[sd_valid, year_index, statistic_index] / exact_sd[sd_valid, year_index, statistic_index]
        rows.append({
            "Year": int(year),
            "Statistic": summary_labels[statistic],
            "Statistic Key": statistic,
            "Parameters Compared": int(valid.sum()),
            "Spearman Correlation": safe_spearman(exact_valid, deeprv_valid),
            "Median Signed Difference": np.median(difference) if valid.any() else np.nan,
            "Median Absolute Difference": np.median(np.abs(difference)) if valid.any() else np.nan,
            "Median Standardised Absolute Mean Difference": np.nanmedian(standardised_difference) if np.isfinite(standardised_difference).any() else np.nan,
            "Median SD Ratio": np.median(sd_ratio) if len(sd_ratio) else np.nan,
        })
accuracy_by_year = pd.DataFrame(rows)
display(accuracy_by_year.round(4))

In [ ]:
accuracy_overall = accuracy_by_year.groupby("Statistic", sort=False).agg(
    Mean_MAD=("Median Absolute Difference", "mean"),
    Mean_MSAD=("Median Standardised Absolute Mean Difference", "mean"),
    Mean_SD_Ratio=("Median SD Ratio", "mean"),
).reset_index()
accuracy_by_year.to_csv(analysis_directory / "accuracy_by_year.csv", index=False)
accuracy_overall.to_csv(analysis_directory / "accuracy_overall.csv", index=False)
display(accuracy_overall.round(4))

In [ ]:
valid = np.isfinite(exact_mean) & np.isfinite(deeprv_mean) & np.isfinite(exact_sd) & (exact_sd > 0) & (exact_count >= minimum_replicates) & (deeprv_count >= minimum_replicates)
absolute_standardised_difference = np.divide(np.abs(deeprv_mean - exact_mean), exact_sd, out=np.full_like(exact_mean, np.nan), where=valid)

colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
plot_percentile = 99

for statistic_index, statistic in enumerate(summary_names):
    all_values = absolute_standardised_difference[:, :, statistic_index]
    finite_values = all_values[np.isfinite(all_values)]
    plot_limit = np.percentile(finite_values, plot_percentile)
    bins = np.linspace(0, plot_limit, 41)

    figure, axes = plt.subplots(2, 5, figsize=(14.4, 6.0), sharex=True, sharey=True, constrained_layout=True)

    for year_index, (year, axis) in enumerate(zip(years, axes.flat)):
        values = all_values[:, year_index]
        values = values[np.isfinite(values)]

        if len(values):
            displayed = values[values <= plot_limit]
            median_value = np.median(values)
            percentage_shown = 100 * len(displayed) / len(values)

            axis.hist(displayed, bins=bins, density=True, color=colors[statistic_index], alpha=0.75, edgecolor="white", linewidth=0.5)
            axis.axvline(median_value, color="#333333", linestyle="--", linewidth=1.2)
            axis.text(0.96, 0.94, f"Median = {median_value:.3f}\n{percentage_shown:.1f}% Within Plot Range", transform=axis.transAxes, ha="right", va="top", fontsize=8)
        else:
            axis.text(0.5, 0.5, "Not Applicable", transform=axis.transAxes, ha="center", va="center", fontsize=9)
        axis.set_title(f"Year {int(year)}", fontsize=11)
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.tick_params(labelsize=9, direction="out")
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    figure.supxlabel("Absolute Standardised Mean Difference", fontsize=14)
    figure.supylabel("Density", fontsize=14)
    figure.suptitle(f"{summary_labels[statistic]} By Year", fontsize=16)
    plt.show()

In [ ]:
selected_statistics = [
    "emd_breeder_next_size_distribution",
    "chi2_total_survivors_by_size_age",
    "chi2_total_breeders_by_size",
    "chi2_total_recruits_by_age",
    "bhattacharyya_nonbreeder_next_size_distribution",
]

def readable_name(name):
    name = name.replace("_", " ").title()
    name = name.replace("Emd", "EMD").replace("Chi2", "Chi-Squared")
    return "\n".join(textwrap.wrap(name, 32))

plt.style.use("seaborn-v0_8-white")
figure, axes = plt.subplots(2, 3, figsize=(10.8, 6.0), constrained_layout=True)

for statistic, axis in zip(selected_statistics, axes.flat):
    statistic_index = np.where(np.asarray(summary_names) == statistic)[0][0]

    exact = exact_mean[:, :, statistic_index]
    deeprv = deeprv_mean[:, :, statistic_index]
    gp_sd = exact_sd[:, :, statistic_index]

    valid = (
        np.isfinite(exact)
        & np.isfinite(deeprv)
        & np.isfinite(gp_sd)
        & (gp_sd > 0)
        & (exact_count[:, :, statistic_index] >= minimum_replicates)
        & (deeprv_count[:, :, statistic_index] >= minimum_replicates)
    )

    standardised_difference = np.abs(deeprv[valid] - exact[valid]) / gp_sd[valid]
    median_value = np.median(standardised_difference)
    upper_limit = max(np.percentile(standardised_difference, 99.5), 1e-12)

    axis.hist(
        standardised_difference,
        bins=60,
        range=(0, upper_limit),
        density=True,
        color="#0072B2",
        alpha=0.85,
    )

    axis.axvline(
        median_value,
        color="#D55E00",
        linewidth=1.4,
        linestyle="--",
    )

    axis.set_title(
        f"{readable_name(statistic)}\nMedian = {median_value:.3f}",
        fontsize=10,
    )

    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

axes.flat[-1].axis("off")

figure.supxlabel("Standardised Absolute Mean Difference", fontsize=14)
figure.supylabel("Probability Density", fontsize=14)
figure.suptitle("Selected Age–Size ABC Statistics: All Years Pooled", fontsize=16)

figure.savefig(analysis_directory / "selected_statistics_standardised_distance_all_years.pdf", bbox_inches="tight")
figure.savefig(analysis_directory / "selected_statistics_standardised_distance_all_years.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

selected_statistics = {
    "emd_breeder_next_size_distribution":
        "EMD\nBreeders' SizeNext Distribution",
    "chi2_total_survivors_by_size_age":
        r"$\chi^2$ Distance" + "\nSurvivors By Size–Age",
    "chi2_total_breeders_by_size":
        r"$\chi^2$ Distance" + "\nBreeders By Size",
    "chi2_total_recruits_by_age":
        r"$\chi^2$ Distance" + "\nRecruits By Age",
    "bhattacharyya_nonbreeder_next_size_distribution":
        "Bhattacharyya Distance\nNon-Breeders' SizeNext",
}

statistic_names = np.asarray(summary_names).astype(str)

plt.style.use("seaborn-v0_8-white")

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

standardised_differences = {}

for statistic in selected_statistics:
    statistic_index = np.where(statistic_names == statistic)[0][0]

    exact = exact_mean[:, :, statistic_index]
    deeprv = deeprv_mean[:, :, statistic_index]
    gp_sd = exact_sd[:, :, statistic_index]

    valid = (
        np.isfinite(exact)
        & np.isfinite(deeprv)
        & np.isfinite(gp_sd)
        & (gp_sd > 0)
        & (exact_count[:, :, statistic_index] >= minimum_replicates)
        & (deeprv_count[:, :, statistic_index] >= minimum_replicates)
    )

    standardised_differences[statistic] = (
        np.abs(deeprv[valid] - exact[valid]) / gp_sd[valid]
    )

upper_limit = max(
    max(
        np.percentile(values, 99)
        for values in standardised_differences.values()
    ),
    1e-12,
)

bin_edges = np.linspace(0, upper_limit, 61)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(5.906, 4.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for axis, (statistic, title) in zip(
    axes.flat,
    selected_statistics.items(),
):
    values = standardised_differences[statistic]
    median_value = np.median(values)

    axis.hist(
        values,
        bins=bin_edges,
        density=True,
        color="#0072B2",
        alpha=0.85,
        edgecolor="none",
    )

    axis.axvline(
        median_value,
        color="#D55E00",
        linewidth=1.5,
        linestyle="--",
    )

    axis.set_title(
    title,
    fontsize=9,
    pad=6,
    linespacing=1.15,
    )

    axis.set_xlim(0, upper_limit)
    axis.tick_params(direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

axes.flat[-1].axis("off")

fig.supxlabel("Standardised Absolute Difference", fontsize=11)
fig.supylabel("Probability Density", fontsize=11)

fig.savefig(analysis_directory / "agesize_selected_statistics_standardised_difference_all_years.pdf", bbox_inches="tight",)
fig.savefig(analysis_directory / "agesize_selected_statistics_standardised_difference_all_years.png", dpi=300, bbox_inches="tight",)

plt.show()

In [ ]:
selected_year = 10
selected_year_index = np.where(years == selected_year)[0][0]
rng = np.random.default_rng(123)
figure, axes = plt.subplots(1, len(summary_names), figsize=(18.0, 3.4), constrained_layout=True)
for statistic_index, (statistic, axis) in enumerate(zip(summary_names, axes)):
    valid = np.isfinite(exact_mean[:, selected_year_index, statistic_index]) & np.isfinite(deeprv_mean[:, selected_year_index, statistic_index]) & (exact_count[:, selected_year_index, statistic_index] >= minimum_replicates) & (deeprv_count[:, selected_year_index, statistic_index] >= minimum_replicates)
    exact_values = exact_mean[:, selected_year_index, statistic_index][valid]
    deeprv_values = deeprv_mean[:, selected_year_index, statistic_index][valid]
    selection = rng.choice(len(exact_values), min(10_000, len(exact_values)), replace=False)
    exact_plot = exact_values[selection]
    deeprv_plot = deeprv_values[selection]
    upper = max(np.percentile(np.concatenate([exact_plot, deeprv_plot]), 99.5), 1e-12)
    correlation = safe_spearman(exact_values, deeprv_values)
    axis.scatter(exact_plot, deeprv_plot, s=5, alpha=0.12, color="#0072B2", rasterized=True)
    axis.plot([0, upper], [0, upper], color="#D55E00", linewidth=1.4)
    axis.set_xlim(0, upper)
    axis.set_ylim(0, upper)
    axis.set_title(f"{summary_labels[statistic]}\nSpearman ρ = {correlation:.3f}", fontsize=9)
    style_axis(axis)
figure.supxlabel("Exact GP Parameter-Level Mean Distance", fontsize=14)
figure.supylabel("DeepRV Parameter-Level Mean Distance", fontsize=14)
figure.suptitle(f"Parameter-Level Distance Agreement: Year {selected_year}", fontsize=16)
figure.savefig(analysis_directory / "parameter_mean_agreement.pdf", bbox_inches="tight")
figure.savefig(analysis_directory / "parameter_mean_agreement.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
rng = np.random.default_rng(123)

for year_index, year in enumerate(years):
    figure, axes = plt.subplots(1, len(summary_names), figsize=(18.0, 3.4), constrained_layout=True)

    for statistic_index, (statistic, axis) in enumerate(zip(summary_names, axes)):
        valid = np.isfinite(exact_mean[:, year_index, statistic_index]) & np.isfinite(deeprv_mean[:, year_index, statistic_index]) & (exact_count[:, year_index, statistic_index] >= minimum_replicates) & (deeprv_count[:, year_index, statistic_index] >= minimum_replicates)
        exact_values = exact_mean[:, year_index, statistic_index][valid]
        deeprv_values = deeprv_mean[:, year_index, statistic_index][valid]
        selection = rng.choice(len(exact_values), min(10_000, len(exact_values)), replace=False)
        exact_plot = exact_values[selection]
        deeprv_plot = deeprv_values[selection]
        upper = max(np.percentile(np.concatenate([exact_plot, deeprv_plot]), 99.5), 1e-12)
        correlation = safe_spearman(exact_values, deeprv_values)
        correlation_label = f"{correlation:.3f}" if np.isfinite(correlation) else "Not Applicable"

        axis.scatter(exact_plot, deeprv_plot, s=5, alpha=0.12, color="#0072B2", rasterized=True)
        axis.plot([0, upper], [0, upper], color="#D55E00", linewidth=1.4)
        axis.set_xlim(0, upper)
        axis.set_ylim(0, upper)
        axis.set_title(f"{summary_labels[statistic]}\nSpearman ρ = {correlation_label}", fontsize=9)
        style_axis(axis)

    figure.supxlabel("Exact GP Parameter-Level Mean Distance", fontsize=14)
    figure.supylabel("DeepRV Parameter-Level Mean Distance", fontsize=14)
    figure.suptitle(f"Parameter-Level Distance Agreement: Year {int(year)}", fontsize=16)
    #figure.savefig(analysis_directory / f"parameter_mean_agreement_year_{int(year)}.pdf", bbox_inches="tight")
    #figure.savefig(analysis_directory / f"parameter_mean_agreement_year_{int(year)}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)